# Домашнее задание на проверку знаний библиотеки для анализа данных Pandas.

Вам представлен датасет, который содержит информацию по результатам сезона 2016 года игры в бейсбол.

Вам необходимо загрузить датасет (он находится в архиве с текущим домашним заданием). Провести его очистку, если это необходимо и ответить на поставленные аналитические
вопросы.

Список колонок, представленных в датасете:

---
* attendance        - количество зрителей матча
* away_team         - название гостевой команды
* away_team_errors  - количество ошибок гостевой команды
* away_team_hits    - количество хитов гостевой команды (хит в бейсболе - удар при котором отбивающий достиг первой базы)
* away_team_runs    - количество ранов гостевой команды (ран в бейсболе - очко заработанное игроком нападения)
* date              - дата проведения матча
* field_type        - тип игрового поля
* game_type         - тип игры
* home_team         - название домашней команды
* home_team_errors  - количество ошибок домашней команды
* home_team_hits    - количество хитов домашней команды (хит в бейсболе - удар при котором отбивающий достиг первой базы)
* home_team_runs    - количество ранов домашней команды (ран в бейсболе - очко заработанное игроком нападения)
* start_time        - время начала игрового матча
* venue             - название места проведения матча (стадион, поле, арена)
* day_of_week       - день недели проведения матча
* temperature       - температура воздуха в день проведения матча в фаренгейтах
* wind_speed        - скорость ветра в день проведения матча
* wind_direction    - направление скорости ветра
* sky               - облачность
* total_runs        - общее количество ранов для двух команд
* game_hours_dec    - длительность матча указаная в часах
* season            - тип игрового сезона
* home_team_win     - результат домашней команды (1 - выигрыш)
* home_team_loss    - результат домашней команды (0 - проигрыш)
* home_team_outcome - исход матча
---

Всего в задании 20 вопросов. За каждый правильный вопрос начисляется 5 баллов. Таким образом за все правильные ответы вы получите - 100 баллов. Оценка затем будет приведена к 10-ти бальной шкале.

---

**Баллы могут быть снижены** за качество кода - старайтесь чтобы код был максимально лаконичным. Названия переменных (если они используются) должны отражать максимальную ее сущность. Вывод старайтесь делать минимальным, так как первичная цель - сделать качественный анализ, чтобы быстро ответить на поставленный вопрос (никаких таблиц на 100 строк).

**Требования по сдаче практического задания**:
1. Ответ на вопрос и текстовый markdown с вашей логикой, почему вы выполнили это действие.
2. Залить в репу на github в папку module_1
3. Ссылку на репо отправлять мне в тг.
4. После дедлайна, неотправленные на почту задания, оцениваются максимум в 7 баллов.

In [2]:
# Импортируйте все необходимые библиотеки и модули которые вам необходимы
import pandas as pd
import numpy as np

In [3]:
# Загрузите датасет и проведите его очистку и форматирование если это необходимо
df = pd.read_csv('baseball_games.csv',
                 index_col=0,
                 parse_dates=['date'],
                 )

# Проводим первичный анализ датасета
print(df.isna().sum()[lambda s: s > 0])   # покажет только проблемные столбцы
print('дубликатов:', df.duplicated().sum())
print('shape:', df.shape)

# По результатам анализа в столбце attendance три Null значения. Было принято решение заполнить нулевые значения средним по столбцу
df['attendance'] = df['attendance'].fillna(df['attendance'].mean())

# Удаляем(если есть) лишние пробелы во всех строковых столбцах
for col in df.select_dtypes(include='str').columns:
    df[col] = df[col].str.strip()

# фильтруем датасет только по regular_season, дальше будем работать с ним
regular = df[df['season'] == 'regular season']

attendance    3
dtype: int64
дубликатов: 0
shape: (2463, 25)


# 1. На какую игру пришло максимальное количество зрителей за весь сезон игр?

In [77]:
regular = df[df['season'] == 'regular season']
mask_att = regular['attendance'] == regular['attendance'].max()
regular.loc[mask_att, ['attendance', 'season', 'date', 'home_team', 'away_team', 'venue', 'season']]

,season,date,home_team,away_team,venue,season
534,regular season,2016-09-20,Los Angeles Dodgers,San Francisco Giants,Dodger Stadium,regular season


Фильтруем таблицу по `df['season']` == `regular season`, ищем игры с макимальным кол-вом зрителей через маску `df['attendance'] == df['attendance'].max()`, через `loc` выводим только основные поля


# 2. Какая игра была самая холодная (temperature) за весь сезон?

In [106]:
mask_temp_min = regular['temperature'] == regular['temperature'].min()
regular.loc[mask_temp_min, ['season', 'date', 'home_team', 'away_team', 'venue', 'temperature']]


,season,date,home_team,away_team,venue,temperature
2409,regular season,2016-04-09,Detroit Tigers,New York Yankees,Comerica Park,31.0


Логика как и в 1 задаче

# 3. Какая игра была самая теплая за весь сезон?

In [69]:
mask_temp_max = regular['temperature'] == regular['temperature'].max()
regular.loc[mask_temp_max, ['season', 'date', 'home_team', 'away_team', 'venue', 'temperature']]

,season,date,home_team,away_team,venue,temperature
2026,regular season,2016-05-13,Arizona Diamondbacks,San Francisco Giants,Chase Field,101.0


Сначала создаём новую таблицу только с `regular season`, далее находим в маске самые теплые игры и с помощью loc выводим игры

# 4. Какая игра в сезоне была самая долгая по продолжительности матча?

In [70]:
mask_hours_max = regular['game_hours_dec'] == regular['game_hours_dec'].max()
regular.loc[mask_hours_max, ['game_hours_dec' ,'season', 'date', 'home_team', 'away_team', 'venue']]

,game_hours_dec,season,date,home_team,away_team,venue
1445,6.216667,regular season,2016-07-01,Toronto Blue Jays,Cleveland Indians,Rogers Centre


Логика как и в задании 3

# 5. Какая игра в сезоне была самая короткая по продолжительности матча?

In [71]:
mask_hours_min = regular['game_hours_dec'] == regular['game_hours_dec'].min()
regular.loc[mask_hours_min, ['game_hours_dec' ,'season', 'date', 'home_team', 'away_team', 'venue']]

,game_hours_dec,season,date,home_team,away_team,venue
423,1.25,regular season,2016-09-29,Pittsburgh Pirates,Chicago Cubs,PNC Park


Логика как ив 3 задаче

# 6. Сколько матчей в сезоне закончилось ничьей?

In [25]:
df['home_team_outcome'].unique()
is_draw = regular['home_team_runs'] == regular['away_team_runs']

print(f'Ничьих: {is_draw.sum()}')

Ничьих: 1


Т.к. в 'home_team_outcome' нету значения Tie, то ничья — матч с равным количеством ранов: `home_team_runs == away_team_runs`. с помощью `df['home_team_runs']` == `df['away_team_runs']` находим такие игры и выводим их кол-во. Кстати это и есть самая короткая игры из 5 задания

# 7. Какая игра была последней в сезоне?

In [105]:
last_day = regular[regular['date'] == regular['date'].max()]
last_day[['season', 'date', 'home_team', 'away_team', 'home_team_runs', 'away_team_runs', 'venue']]

,season,date,home_team,away_team,home_team_runs,away_team_runs,venue
349,regular season,2016-10-02,Washington Nationals,Miami Marlins,10,7,Nationals Park
385,regular season,2016-10-02,Texas Rangers,Tampa Bay Rays,4,6,Globe Life Park in Arlington
386,regular season,2016-10-02,St. Louis Cardinals,Pittsburgh Pirates,10,4,Busch Stadium III
387,regular season,2016-10-02,San Francisco Giants,Los Angeles Dodgers,7,1,AT&T Park
388,regular season,2016-10-02,Seattle Mariners,Oakland Athletics,2,3,Safeco Field
389,regular season,2016-10-02,Philadelphia Phillies,New York Mets,5,2,Citizens Bank Park
390,regular season,2016-10-02,New York Yankees,Baltimore Orioles,2,5,Yankee Stadium III
391,regular season,2016-10-02,Kansas City Royals,Cleveland Indians,2,3,Kauffman Stadium
392,regular season,2016-10-02,Colorado Rockies,Milwaukee Brewers,4,6,Coors Field
393,regular season,2016-10-02,Cincinnati Reds,Chicago Cubs,4,7,Great American Ball Park


Фильтруем датасет по сезоны и находим последнюю дату c помощью маски. В выводе 15 игр в один день. Можно еще отфильтровать по `start_time`, но по-моему это избыток для данной задачи

# 8. У какой игры было минимальное количество зрителей?

In [104]:
mask_att_min = regular['attendance'] == regular['attendance'].min()
regular.loc[mask_att_min, ['attendance', 'season', 'date', 'home_team', 'away_team', 'venue']]

,attendance,season,date,home_team,away_team,venue,season
2130,8766.0,regular season,2016-05-04,Cleveland Indians,Detroit Tigers,Progressive Field,regular season


То же, что в задании 1, но ищем минимум

# 9. Какая игра в сезоне была самая ветренная?

In [109]:
windy_game = regular[regular['wind_speed'] == regular['wind_speed'].max()]
windy_game[['wind_speed', 'date', 'home_team', 'away_team', 'home_team_runs', 'away_team_runs', 'venue']]

,wind_speed,date,home_team,away_team,home_team_runs,away_team_runs,venue
1655,25.0,2016-06-13,San Francisco Giants,Milwaukee Brewers,11,5,AT&T Park
2005,25.0,2016-05-15,Boston Red Sox,Houston Astros,10,9,Fenway Park


Самая ветренная игра - это игра с самым сильным ветром, фильтруем датасет по маске и выводим игры с максимальным ветром

# 10. В какой игре получили максимальное количество очков?

In [110]:
max_runs_game = regular[regular['total_runs'] == regular['total_runs'].max()]
max_runs_game[['total_runs', 'date', 'home_team', 'away_team', 'home_team_runs', 'away_team_runs', 'venue']]

,total_runs,date,home_team,away_team,home_team_runs,away_team_runs,venue
1788,29,2016-06-02,San Diego Padres,Seattle Mariners,13,16,Petco Park


Максимальное кол-во очков в игре - это игра с максимальным кол-вом ранов за обе команды

# 11. Какая игра содержала максимальное количество ошибок домашней команды?

In [111]:
max_errors_game = regular[regular['home_team_errors'] == regular['home_team_errors'].max()]
max_errors_game[['home_team_errors', 'date', 'home_team', 'away_team', 'home_team_runs', 'away_team_runs', 'venue']]

,home_team_errors,date,home_team,away_team,home_team_runs,away_team_runs,venue
1178,5,2016-07-27,Milwaukee Brewers,Arizona Diamondbacks,1,8,Miller Park


Максимальное количество ошибок домашней команды - это максимальное кол-во `home_team_errors`

# 12. В какой игре было максимальное количество хитов?

In [23]:
hits = regular.assign(total_hits=regular['home_team_hits'] + regular['away_team_hits'])
is_max_hits = hits['total_hits'] == hits['total_hits'].max()

hits.loc[is_max_hits, ['total_hits', 'date', 'home_team', 'away_team', 'home_team_hits', 'away_team_hits', 'venue']]

,total_hits,date,home_team,away_team,home_team_hits,away_team_hits,venue
1413,37,2016-07-04,Boston Red Sox,Texas Rangers,21,16,Fenway Park


Т.к. готового `total_hits` в датасете нет, находим максимальную сумму `home_team_hits` и `away_team_hits`

# 13. Выведите количество игр которая сыграла каждая команда в данном сезоне?

In [4]:
games_per_team = pd.concat([regular['home_team'], regular['away_team']]).value_counts()
games_per_team

Kansas City Royals               162
Cincinnati Reds                  162
Baltimore Orioles                162
Arizona Diamondbacks             162
Texas Rangers                    162
Tampa Bay Rays                   162
San Diego Padres                 162
Pittsburgh Pirates               162
Oakland Athletics                162
New York Yankees                 162
Milwaukee Brewers                162
Seattle Mariners                 162
Philadelphia Phillies            162
New York Mets                    162
Minnesota Twins                  162
Houston Astros                   162
Chicago Cubs                     162
Colorado Rockies                 162
Washington Nationals             162
Toronto Blue Jays                162
St. Louis Cardinals              162
San Francisco Giants             162
Chicago White Sox                162
Boston Red Sox                   162
Los Angeles Dodgers              162
Los Angeles Angels of Anaheim    162
Atlanta Braves                   161
M

Сливаем два столбца в одну серию, т.к. в каждой игре команда либо домашняя либо гость, поэтому объединяем их в единое с помощью 'concat' и считаем кол-во значений с помощью `value_count`

# 14. Какая команда выиграла наибольшое количество матчей в сезоне? (Будьте внимательны с типом матча в сезоне - season).

In [30]:
home_wins = regular.loc[regular['home_team_win'] == 1, 'home_team'].value_counts()
away_wins = regular.loc[regular['home_team_loss'] == 1, 'away_team'].value_counts()

wins = home_wins.add(away_wins, fill_value=0).astype(int).sort_values(ascending=False)
print(f'Лидеров: {(wins == wins.max()).sum()}, максимум побед: {wins.max()}')
wins.head(1)

Лидеров: 1, максимум побед: 103


home_team
Chicago Cubs    103
Name: count, dtype: int64

Победа определяется по флагам: если `home_team_win == 1` — победили хозяева (победа команде из `home_team`), если `home_team_loss == 1` — проиграли хозяева, значит победили гости (победа команде из `away_team`). Находим все победы команд помощью `value_count`, далее складываем две Series, выводим команду-лидера



# 15. Какая команда выиграла наибольшее количество домашних матчей в сезоне?

In [31]:
home_team_wins = regular.loc[regular['home_team_win'] == 1, 'home_team'].value_counts()
print(f'Лидеров: {(home_team_wins == home_team_wins.max()).sum()}, максимум побед: {home_team_wins.max()}')
home_team_wins.head(1)

Лидеров: 1, максимум побед: 57


home_team
Chicago Cubs    57
Name: count, dtype: int64

То же самое, что и в 14 задаче, только ищём только по `home_team_win`

# 16. Какая команда выиграла наибольшее количество гостевых матчей в сезоне?

In [35]:
away_team_wins = regular.loc[regular['home_team_loss'] == 1, 'away_team'].value_counts()
print(f'Лидеров: {(away_team_wins == away_team_wins.max()).sum()}, максимум побед: {away_team_wins.max()}')
away_team_wins.head(1)

Лидеров: 2, максимум побед: 52


away_team
Minnesota Twins    52
Name: count, dtype: int64

То же самое, что и в 15 задаче, только смотрим `home_team_loss == 1`, то есть проигрыш домашней команды

# 17. Какая команда проиграла наибольшее количество матчей в сезоне?

In [36]:
home_losses = regular.loc[regular['home_team_win'] == 0, 'home_team'].value_counts()
away_losses = regular.loc[regular['home_team_loss'] == 0, 'away_team'].value_counts()

losses = home_losses.add(away_losses, fill_value=0).astype(int).sort_values(ascending=False)
print(f'Лидеров: {(losses == losses.max()).sum()}, максимум проигрышей: {losses.max()}')
losses.head(1)

Лидеров: 1, максимум проигрышей: 103


home_team
Minnesota Twins    103
Name: count, dtype: int64

# 18. Зависит ли выигрыш от количества посетителей матча? (https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.cov.html)

In [37]:
pairs = df[['attendance', 'home_team_win']]
print('Ковариация:\n', pairs.cov())
print('Корреляция Пирсона:', round(df['attendance'].corr(df['home_team_win']), 4))

Ковариация:
                  attendance  home_team_win
attendance     9.740601e+07     224.697740
home_team_win  2.246977e+02       0.249235
Корреляция Пирсона: 0.0456


Вопрос «зависит ли выигрыш от посещаемости» — это вопрос о наличии связи между двумя числовыми признаками, поэтому считаю ковариацию (`DataFrame.cov`) и корреляцию Пирсона. cov(attendance, home_team_win) = 224.7 — число в единицах «зрители × флаг», само по себе не интерпретируется, поэтому решающая метрика — корреляция: r = 0.046, что практически ноль (|r| < 0.1 — связи нет). Вывод: зависимости выигрыша от количества зрителей нет.


# 19. Правда ли что большинство проигрышных домашних матчей приходятся на Субботу и Воскресенье?

In [9]:
home_losses = df[df['home_team_loss'] == 1]

weekend_share = home_losses['day_of_week'].isin(['Saturday', 'Sunday']).mean()
print(f'Домашних поражений в Сб+Вс: {weekend_share:.1%}')

Домашних поражений в Сб+Вс: 31.5%


Нет, утверждение неверно. На выходные приходится 31.5% домашних поражений, это не большинство. Логика программы следующая: находим все матчи где `home_team_loss == 1` и считаем процент поражений в выходные дни

# 20. Правда ли что наибольшее количество ранов происходит в холодную погоду? (Холодной погодой считается погода ниже 0 градусов)

In [13]:
print('Мин. температура:', df['temperature'].min(), '°F | игр ниже 0:', int((df['temperature'] < 0).sum()))

# 2) в какой температуре случился рекорд ранов
top = df.loc[df['total_runs'].idxmax()]
print(f'Рекорд ранов: {top["total_runs"]} при {top["temperature"]} °F ({top["date"].date()})')

# 3) связь температуры и ранов в целом
print('Corr temperature-total_runs:', round(df['temperature'].corr(df['total_runs']), 3))

Мин. температура: 31.0 °F | игр ниже 0: 0
Рекорд ранов: 29 при 76.0 °F (2016-06-02)
Corr temperature-total_runs: 0.082


`temperature` в датасете измеряется в Фаренгейтах, минимум — 31 °F, поэтому игр с температурой ниже 0 °F нет ни одной, и условие вопроса не выполняется. Сезонный рекорд ранов (29) пришёлся на игру при 76 °F, то есть на тёплую погоду. Корреляция `temperature` и `total_runs` = 0.08 — связи практически нет. Если понимать «ниже 0 градусов» как 0 °C (32 °F), под условие попадает единственная игра (31 °F, 12 ранов), что тоже не максимум. Вывод: утверждение неверно.
